In [7]:
pip install -U openai-whisper


In [8]:
# pip install --no-deps -U yt-dlp


In [9]:
# import os
# import yt_dlp

# url = "https://www.youtube.com/watch?v=Fm1IMtsTD1g"

# # Define your target directory path
# # Use forward slashes (/) or raw strings r"C:\path\to\folder" on Windows
# save_path = "input"

# ydl_opts = {
#     'format': 'bestvideo[height<=480][ext=mp4]+bestaudio[ext=m4a]/best[height<=480][ext=mp4]/best',
#     'merge_output_format': 'mp4',

#     # Combines the folder path and the video title
#     'outtmpl': os.path.join(save_path, '%(title)s.%(ext)s'),
# }

# with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#      # 1. Extract metadata without downloading yet
#     info = ydl.extract_info(url, download=False)

#     # 2. Predict the exact filename based on your options
#     predicted_path = ydl.prepare_filename(info)
#     print(f"The file will be saved to: {predicted_path}")

#     # 3. Proceed to download
#     ydl.download([url])


In [ ]:

import whisper

# Load model
model = whisper.load_model("base")

# Transcribe speech directly from the video
result = model.transcribe("/content/YTDown.com_YouTube_Media_sNVprF30zXk_English-Podcast-for-Learning-English-B2-C1-Health-and-Wellness_006_144p (1).mp4",
                          word_timestamps = True)

print("Detected language:", result["language"])
print("\nTrans  cript:")
print(result)

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 53.2MiB/s]
/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [ ]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

def combine_words_into_sentences(result):
    # Collect words and their timestamps across all segments
    words = []

    for segment in result["segments"]:
        for word in segment.get("words", []):
            words.append({
                "word": word["word"],
                "start": word["start"],
                "end": word["end"],
                "id": segment["id"]
            })

    # Reconstruct the complete transcript
    text = "".join(item["word"] for item in words)

    # Get sentence boundaries using NLTK
    sentences_text = sent_tokenize(text)

    sentences = []
    char_position = 0

    for sentence_text in sentences_text:
        # Find this sentence in the reconstructed transcript
        start_char = text.find(sentence_text, char_position)

        if start_char == -1:
            continue

        end_char = start_char + len(sentence_text)
        char_position = end_char

        # Map sentence character positions to Whisper words
        sentence_words = []
        current_char = 0

        for item in words:
            word_start = current_char
            word_end = word_start + len(item["word"])
            current_char = word_end

            if word_end > start_char and word_start < end_char:
                sentence_words.append(item)

        if sentence_words:
            sentences.append({
                "text": sentence_text.strip(),
                "start": sentence_words[0]["start"],
                "end": sentence_words[-1]["end"]
            })

    return sentences


sentences = combine_words_into_sentences(result)
# print(sentences)
# for i, sentence in enumerate(sentences, 1):
#     print(
#         f'{i}. [{sentence["start"]:.2f}s - '
#         f'{sentence["end"]:.2f}s] {sentence["text"]}'
#     )


In [ ]:

import nltk
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.tokenize import sent_tokenize

nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")
from moviepy.editor import VideoFileClip


QUESTION_WORDS = {
    "what", "why", "where", "when",
    "who", "whom", "whose", "which", "how"
}

AUXILIARY_VERBS = {
    "am", "is", "are", "was", "were",
    "do", "does", "did", "have", "has", "had",
    "can", "could", "will", "would",
    "shall", "should", "may", "might", "must"
}

AGREE_WORDS = {
    "agree", "agreed", "agreeing",
    "right", "exactly", "absolutely",
    "correct", "true", "indeed",
    "definitely", "certainly"
}

DISAGREE_WORDS = {
    "disagree", "disagreed", "disagreeing",
    "wrong", "incorrect", "false",
    "no", "never", "not"
}

def detect_question(sentence):
    if not sentence:
        return False

    # Rule 1: Question mark
    if "?" in sentence:
        return True

    words = word_tokenize(sentence)
    if not words:
        return False

    first_word = words[0].lower()

    # Rule 2: Question word at the beginning
    if first_word in QUESTION_WORDS:
        return True

    # Rule 3: Auxiliary verb at the beginning
    if first_word in AUXILIARY_VERBS:
        return True

    return False

def detect_agreement_disagreement(sentence):
    if not sentence:
        return False

    words = AGREE_WORDS | DISAGREE_WORDS

    return any(word in sentence.lower() for word in words)


segments = sentences

for i, segment in enumerate(segments):
    sentence = segment["text"]
    # print(sentence, "->", detect_question(sentence))
    if(detect_question(sentence) or detect_agreement_disagreement(sentence)): #if its a question


      # print(segment['seek'])

      start_time = segment['start']
      end_time = segment['end']
      #calculate duration
      duration = segment['end'] - segment['start']
      # print("Duration", duration)

      print("Start time", start_time)
      # print("End time", length)
      # print(sentence)
      # cut videos from start to end and save
      video_output  = 'output_video_' + str(i) + '.mp4'
      print(video_output)
      video = VideoFileClip("/content/YTDown.com_YouTube_Media_sNVprF30zXk_English-Podcast-for-Learning-English-B2-C1-Health-and-Wellness_006_144p (1).mp4")
      end_time = min(max(start_time + 60, end_time), video.duration)
      clip = video.subclip(start_time, end_time)

      clip.write_videofile(video_output, codec="libx264", audio_codec="aac")
      clip.close()
      video.close()
      print("\n")

In [ ]:
# import os
# import glob
# #remove files
# files = glob.glob('*.mp4')
# for f in files:
#     os.remove(f)
